In [10]:
import os
import torch
import numpy as np
from PIL import Image
from collections import Counter
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import euclidean_distances

from semseg.models.subclass_segformer_unified import SubclassSegFormer_Unified

# ================= 設定區域 =================
TXT_BASE_DIR = r"/folds_experiment/fold_0"
FIELDS_TO_TEST = ['F', 'G', 'H', 'I', 'J']

MEDOID_PATH = "prompt_256D/visual_bases_K6_medoid.pth"
PRETRAINED_WEIGHT_PATH = r"/output_mix5_SubclassSegFormer_Unified/SubclassSegFormer_Unified_MiT-B0_ripple.pth"

# 💡 核心修正：嚴格限制相似度必須大於 80% (或距離小於 0.63) 才算認得，否則直接算 OOD
SIMILARITY_THRESHOLD = 0.6

IMAGE_SIZE = [800, 800]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# ============================================

def load_custom_model(weight_path):
    model = SubclassSegFormer_Unified(backbone='MiT-B0', num_classes=2, subclass=64, num_prompts=5)
    if os.path.exists(weight_path):
        checkpoint = torch.load(weight_path, map_location='cpu', weights_only=False)
        state_dict = checkpoint.get('state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint.state_dict()
        q_prime_key = 'decode_head.subclass_block.q_prime'
        if q_prime_key in state_dict:
            if state_dict[q_prime_key].shape != model.decode_head.subclass_block.q_prime.shape:
                del state_dict[q_prime_key]
        model.load_state_dict(state_dict, strict=False)
    model = model.to(DEVICE).eval()
    return model

def process_field(txt_path, model, medoids):
    new_features_list = []
    with open(txt_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    for line in lines:
        if not line.strip(): continue
        file_path = line.split('|')[1].replace('PATH:', '').strip()
        if not os.path.exists(file_path): continue

        img_pil = Image.open(file_path).convert('RGB')
        image = TF.to_tensor(img_pil)
        image = TF.resize(image, IMAGE_SIZE, interpolation=TF.InterpolationMode.BILINEAR)
        image = TF.normalize(image, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            features = model.backbone(image)
            seg_map = model.decode_head.get_fusion_map(features)
        feature_map = seg_map.squeeze(0).cpu()

        label_np = np.array(Image.open(file_path).convert('L'))
        label_tensor = torch.tensor((label_np > 0).astype(np.uint8)).unsqueeze(0).unsqueeze(0).float()
        label_resized = F.interpolate(label_tensor, size=feature_map.shape[1:], mode='nearest').squeeze().long()
        ripple_mask = (label_resized == 1)

        if ripple_mask.sum() == 0: continue
        new_features_list.append(feature_map[:, ripple_mask].mean(dim=1).numpy())

    if len(new_features_list) == 0: return "無有效水花", 0.0

    new_features_norm = normalize(np.vstack(new_features_list), norm='l2', axis=1)
    distances = euclidean_distances(new_features_norm, medoids)
    cosine_sims = 1 - (distances ** 2) / 2

    # 判定每張圖的歸屬
    votes = []
    winner_scores = []

    for sim in cosine_sims:
        best_c = np.argmax(sim)
        best_score = sim[best_c]

        # 核心過濾：如果最高相似度連設定門檻都不到，直接投給 OOD (都不像)
        if best_score < SIMILARITY_THRESHOLD:
            votes.append("OOD")
            winner_scores.append(best_score) # 紀錄它本來的最高分供參考
        else:
            votes.append(best_c)
            winner_scores.append(best_score)

    # 眾數投票
    vote_counts = Counter(votes)
    final_decision = vote_counts.most_common(1)[0][0]

    # 計算該歸屬群集的平均分數
    indices = [i for i, v in enumerate(votes) if v == final_decision]
    avg_score_pct = np.mean([winner_scores[idx] for idx in indices]) * 100

    if final_decision == "OOD":
        return "OOD (都不像，建議新增 Cluster)", avg_score_pct
    else:
        return f"Cluster {final_decision}", avg_score_pct

def run_batch_inference():
    model = load_custom_model(PRETRAINED_WEIGHT_PATH)
    medoids = torch.load(MEDOID_PATH, map_location='cpu').numpy()

    print("\n" + "="*60)
    print("                五大場域判定結果 (引入 OOD 門檻)")
    print("="*60)
    for field in FIELDS_TO_TEST:
        txt_path = os.path.join(TXT_BASE_DIR, f"{field}_mask_hard_samples_val_list.txt")
        if not os.path.exists(txt_path):
            print(f"場域 [{field}]: 找不到清單檔案")
            continue

        res_tag, avg_score = process_field(txt_path, model, medoids)
        print(f"場域 [{field}]: 判定結果 -> {res_tag} | 平均分數: {avg_score:.1f}%")
    print("="*60)

if __name__ == "__main__":
    run_batch_inference()


                五大場域判定結果 (引入 OOD 門檻)
場域 [F]: 判定結果 -> Cluster 4 | 平均分數: 69.2%
場域 [G]: 判定結果 -> Cluster 4 | 平均分數: 68.6%
場域 [H]: 判定結果 -> Cluster 4 | 平均分數: 67.9%
場域 [I]: 判定結果 -> Cluster 4 | 平均分數: 68.5%
場域 [J]: 判定結果 -> Cluster 4 | 平均分數: 69.5%


In [11]:
import os
import torch
import numpy as np
from PIL import Image
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import euclidean_distances
from semseg.models.subclass_segformer_unified import SubclassSegFormer_Unified

# ================= 填入你的路徑 =================
TEST_IMG_PATH = r"C:\Users\user\PycharmProjects\meta_expand_test\IMG6235\label\IMG_6235_cut_4.jpg"
MEDOID_PATH = "prompt_256D/visual_bases_K6_medoid.pth"
PRETRAINED_WEIGHT_PATH = r"/output_mix5_SubclassSegFormer_Unified/SubclassSegFormer_Unified_MiT-B0_ripple.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# ===============================================

print(">>> 載入模型...")
model = SubclassSegFormer_Unified(backbone='MiT-B0', num_classes=2, subclass=64, num_prompts=5)
checkpoint = torch.load(PRETRAINED_WEIGHT_PATH, map_location='cpu', weights_only=False)
state_dict = checkpoint.get('state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint.state_dict()
if 'decode_head.subclass_block.q_prime' in state_dict:
    del state_dict['decode_head.subclass_block.q_prime']
model.load_state_dict(state_dict, strict=False)
model = model.to(DEVICE).eval()

medoids = torch.load(MEDOID_PATH, map_location='cpu').numpy()

# 影像前處理
img_pil = Image.open(TEST_IMG_PATH).convert('RGB')
image = TF.to_tensor(img_pil)
image = TF.resize(image, [800, 800], interpolation=TF.InterpolationMode.BILINEAR)
image = TF.normalize(image, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    features = model.backbone(image)
    seg_map = model.decode_head.get_fusion_map(features)
feature_map = seg_map.squeeze(0).cpu()

label_np = np.array(Image.open(TEST_IMG_PATH).convert('L'))
label_tensor = torch.tensor((label_np > 0).astype(np.uint8)).unsqueeze(0).unsqueeze(0).float()
label_resized = F.interpolate(label_tensor, size=feature_map.shape[1:], mode='nearest').squeeze().long()
ripple_mask = (label_resized == 1)

gap_feature = feature_map[:, ripple_mask].mean(dim=1).numpy().reshape(1, -1)
new_features_norm = normalize(gap_feature, norm='l2', axis=1)

distances = euclidean_distances(new_features_norm, medoids)
cosine_sims = 1 - (distances ** 2) / 2

print("\n=== 🎯 單圖診斷結果 ===")
print("圖片與 6 個 Cluster 的 Cosine 相似度：")
for i, sim in enumerate(cosine_sims[0]):
    print(f"Cluster {i}: {sim*100:.2f}%")

>>> 載入模型...

=== 🎯 單圖診斷結果 ===
圖片與 6 個 Cluster 的 Cosine 相似度：
Cluster 0: 65.12%
Cluster 1: 45.24%
Cluster 2: 33.68%
Cluster 3: 47.53%
Cluster 4: 68.93%
Cluster 5: 45.90%


In [12]:
import os
import torch
import numpy as np
from PIL import Image
from pathlib import Path
from collections import Counter
from tqdm import tqdm
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import euclidean_distances

# 確保能從專案中正確匯入模型
from semseg.models.subclass_segformer_unified import SubclassSegFormer_Unified

# ================= 設定區域 =================
# 1. 訓練場域 (A-E) 原始路徑
TRAIN_ROOT_PATH = r"C:\Users\user\PycharmProjects\organized_ripple_4fold"
TRAIN_FIELDS = ['barrel', 'sea', 'LNG', 'seabass_hmh', 'noon_jsj']

# 2. 測試場域 (F-J) 列表路徑與設定
TXT_BASE_DIR = r"/folds_experiment/fold_0"
FIELDS_TO_TEST = ['F', 'G', 'H', 'I', 'J']

# 3. 模型與超參數
PRETRAINED_WEIGHT_PATH = r"/output_mix5_SubclassSegFormer_Unified/SubclassSegFormer_Unified_MiT-B0_ripple.pth"
MEDOID_SAVE_DIR = "prompt_256D"

RANGE_N_CLUSTERS = [6, 8, 10, 12, 16] # K 值搜尋範圍
SIMILARITY_THRESHOLD = 0.80 # OOD 判定門檻
IMAGE_SIZE = [800, 800]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# ============================================

def load_custom_model(weight_path):
    print(">>> 載入統一版模型骨架 (SubclassSegFormer_Unified)...")
    model = SubclassSegFormer_Unified(backbone='MiT-B0', num_classes=2, subclass=64, num_prompts=5)
    if os.path.exists(weight_path):
        checkpoint = torch.load(weight_path, map_location='cpu', weights_only=False)
        state_dict = checkpoint.get('state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint.state_dict()
        q_prime_key = 'decode_head.subclass_block.q_prime'
        if q_prime_key in state_dict:
            if state_dict[q_prime_key].shape != model.decode_head.subclass_block.q_prime.shape:
                del state_dict[q_prime_key]
        model.load_state_dict(state_dict, strict=False)
        print(f"✅ 模型權重載入完成: {weight_path}")
    model = model.to(DEVICE).eval()
    return model

def extract_gap_feature(img_path, lbl_path, model):
    """單張影像特徵提取與 GAP"""
    img_pil = Image.open(img_path).convert('RGB')
    image = TF.to_tensor(img_pil)
    image = TF.resize(image, IMAGE_SIZE, interpolation=TF.InterpolationMode.BILINEAR)
    image = TF.normalize(image, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        features = model.backbone(image)
        seg_map = model.decode_head.get_fusion_map(features)
    feature_map = seg_map.squeeze(0).cpu() # (256, 200, 200)

    label_np = np.array(Image.open(lbl_path).convert('L'))
    label_tensor = torch.tensor((label_np > 0).astype(np.uint8)).unsqueeze(0).unsqueeze(0).float()
    label_resized = F.interpolate(label_tensor, size=feature_map.shape[1:], mode='nearest').squeeze().long()
    ripple_mask = (label_resized == 1)

    if ripple_mask.sum() == 0:
        return None
    return feature_map[:, ripple_mask].mean(dim=1).numpy()

def step1_build_knowledge_base(model):
    print("\n" + "="*50)
    print(" 步驟 1/3：提取 A-E 訓練集特徵 (建立最新知識庫)")
    print("="*50)

    all_features_list = []

    for field in TRAIN_FIELDS:
        img_dir = Path(TRAIN_ROOT_PATH) / field / 'images'
        lbl_dir = Path(TRAIN_ROOT_PATH) / field / 'labels_detectron2'

        if not img_dir.exists() or not lbl_dir.exists():
            print(f"⚠️ 跳過場域 {field}: 找不到資料夾")
            continue

        img_files = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
        print(f"正在處理 {field} ({len(img_files)} 張)...")

        for img_path in tqdm(img_files, leave=False):
            lbl_path = lbl_dir / f"{img_path.stem}.png"
            if not lbl_path.exists(): continue

            feat = extract_gap_feature(img_path, lbl_path, model)
            if feat is not None:
                all_features_list.append(feat)

    all_features_np = np.vstack(all_features_list)
    print(f"✅ A-E 特徵提取完成！共收集 {len(all_features_list)} 筆有效水花特徵。")
    return all_features_np

def step2_run_kmeans(features_np):
    print("\n" + "="*50)
    print(" 步驟 2/3：自動尋找最佳 K 值 (Silhouette) 與質心提取")
    print("="*50)

    features_norm = normalize(features_np, norm='l2', axis=1)

    best_k = RANGE_N_CLUSTERS[0]
    best_score = -1

    # 抽樣計算輪廓係數以加速
    sample_size = min(10000, features_norm.shape[0])
    indices = np.random.choice(features_norm.shape[0], sample_size, replace=False)
    sample_data = features_norm[indices]

    for n_clusters in RANGE_N_CLUSTERS:
        clusterer = KMeans(n_clusters=n_clusters, random_state=3407, n_init='auto')
        cluster_labels = clusterer.fit_predict(sample_data)

        silhouette_avg = silhouette_score(sample_data, cluster_labels)
        print(f"For n_clusters = {n_clusters}, Silhouette Score : {silhouette_avg:.4f}")

        if silhouette_avg > best_score:
            best_score = silhouette_avg
            best_k = n_clusters

    print(f"===> 最佳 K 值為: {best_k} (Score: {best_score:.4f})")

    print(f"\n執行最終 K-Means (K={best_k}) 並提取代表點...")
    kmeans = KMeans(n_clusters=best_k, random_state=3407, n_init=10)
    labels = kmeans.fit_predict(features_norm)

    medoids = []
    for i in range(best_k):
        cluster_points = features_norm[np.where(labels == i)[0]]
        centroid = kmeans.cluster_centers_[i].reshape(1, -1)
        min_dist_idx = np.argmin(euclidean_distances(cluster_points, centroid))
        medoids.append(cluster_points[min_dist_idx])

    medoids_tensor = torch.tensor(np.array(medoids)).float()

    save_path = os.path.join(MEDOID_SAVE_DIR, f"visual_bases_K{best_k}_medoid_NEW.pth")
    os.makedirs(MEDOID_SAVE_DIR, exist_ok=True)
    torch.save(medoids_tensor, save_path)

    print(f"✅ 質心重新計算完畢！已儲存至: {save_path}")
    return medoids_tensor.numpy(), best_k

def step3_test_new_fields(model, medoids, best_k):
    print("\n" + "="*50)
    print(" 步驟 3/3：測試 F-J 新場域並進行 OOD 判定")
    print("="*50)

    for field in FIELDS_TO_TEST:
        txt_path = os.path.join(TXT_BASE_DIR, f"{field}_mask_hard_samples_val_list.txt")
        if not os.path.exists(txt_path):
            print(f"場域 [{field}]: 找不到清單檔案，跳過。")
            continue

        features_list = []
        with open(txt_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        for line in lines:
            if not line.strip(): continue
            file_path = line.split('|')[1].replace('PATH:', '').strip()
            if not os.path.exists(file_path): continue

            feat = extract_gap_feature(file_path, file_path, model) # Label 跟原圖在同一路徑
            if feat is not None:
                features_list.append(feat)

        if not features_list:
            print(f"場域 [{field}]: 判定結果 -> 無有效水花數據")
            continue

        features_norm = normalize(np.vstack(features_list), norm='l2', axis=1)
        distances = euclidean_distances(features_norm, medoids)
        cosine_sims = 1 - (distances ** 2) / 2

        votes = []
        winner_scores = []

        for sim in cosine_sims:
            best_c = np.argmax(sim)
            best_score = sim[best_c]

            if best_score < SIMILARITY_THRESHOLD:
                votes.append("OOD")
            else:
                votes.append(best_c)
            winner_scores.append(best_score)

        vote_counts = Counter(votes)
        final_decision = vote_counts.most_common(1)[0][0]

        indices = [i for i, v in enumerate(votes) if v == final_decision]
        avg_score_pct = np.mean([winner_scores[idx] for idx in indices]) * 100

        if final_decision == "OOD":
            print(f"場域 [{field}]: 判定結果 -> OOD (都不像，建議新增 Cluster) | 原始最高均分: {avg_score_pct:.1f}%")
        else:
            print(f"場域 [{field}]: 判定結果 -> Cluster {final_decision} | 平均分數: {avg_score_pct:.1f}%")

if __name__ == "__main__":
    model = load_custom_model(PRETRAINED_WEIGHT_PATH)

    # 步驟 1
    features_ae = step1_build_knowledge_base(model)

    # 步驟 2
    new_medoids, best_k = step2_run_kmeans(features_ae)

    # 步驟 3
    step3_test_new_fields(model, new_medoids, best_k)

>>> 載入統一版模型骨架 (SubclassSegFormer_Unified)...
✅ 模型權重載入完成: C:\Users\user\PycharmProjects\subclass_segformer\output_mix5_SubclassSegFormer_Unified\SubclassSegFormer_Unified_MiT-B0_ripple.pth

 步驟 1/3：提取 A-E 訓練集特徵 (建立最新知識庫)
正在處理 barrel (500 張)...


正在處理 sea (500 張)...


正在處理 LNG (500 張)...


正在處理 seabass_hmh (500 張)...


正在處理 noon_jsj (500 張)...


✅ A-E 特徵提取完成！共收集 2500 筆有效水花特徵。

 步驟 2/3：自動尋找最佳 K 值 (Silhouette) 與質心提取


C:\Users\user\anaconda3\envs\subclass_5090\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=10.
  warnings.warn(
C:\Users\user\anaconda3\envs\subclass_5090\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=10.
  warnings.warn(
C:\Users\user\anaconda3\envs\subclass_5090\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=10.
  warnings.warn(


For n_clusters = 6, Silhouette Score : 0.2926
For n_clusters = 8, Silhouette Score : 0.2712
For n_clusters = 10, Silhouette Score : 0.2477


C:\Users\user\anaconda3\envs\subclass_5090\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=10.
  warnings.warn(
C:\Users\user\anaconda3\envs\subclass_5090\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=10.
  warnings.warn(
C:\Users\user\anaconda3\envs\subclass_5090\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=10.
  warnings.warn(


For n_clusters = 12, Silhouette Score : 0.2148
For n_clusters = 16, Silhouette Score : 0.2202
===> 最佳 K 值為: 6 (Score: 0.2926)

執行最終 K-Means (K=6) 並提取代表點...
✅ 質心重新計算完畢！已儲存至: prompt/prompt_256D\visual_bases_K6_medoid_NEW.pth

 步驟 3/3：測試 F-J 新場域並進行 OOD 判定
場域 [F]: 判定結果 -> OOD (都不像，建議新增 Cluster) | 原始最高均分: 32.8%
場域 [G]: 判定結果 -> OOD (都不像，建議新增 Cluster) | 原始最高均分: 32.0%
場域 [H]: 判定結果 -> OOD (都不像，建議新增 Cluster) | 原始最高均分: 28.0%
場域 [I]: 判定結果 -> OOD (都不像，建議新增 Cluster) | 原始最高均分: 30.0%
場域 [J]: 判定結果 -> OOD (都不像，建議新增 Cluster) | 原始最高均分: 36.4%


In [ ]:
import os
import json
import torch
import numpy as np
from PIL import Image, ImageDraw
from collections import Counter
from pathlib import Path
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision.utils import make_grid, save_image
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import euclidean_distances

from semseg.models.subclass_segformer_unified import SubclassSegFormer_Unified

# ================= 設定區域 =================
TEST_TXT_PATH = r"/folds_experiment/fold_0/F_mask_hard_samples_val_list.txt"
PRETRAINED_WEIGHT_PATH = r"/output_mix5_SubclassSegFormer_Unified/SubclassSegFormer_Unified_MiT-B0_ripple.pth"
MEDOID_PATH = "prompt_256D/visual_bases_K6_medoid_NEW.pth"
OUTPUT_DIR = "vlm_knowledge_construction_new_cluster"

os.makedirs(OUTPUT_DIR, exist_ok=True)

SIMILARITY_THRESHOLD = 0.80
IMAGE_SIZE = [800, 800]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(">>> 載入模型...")
model = SubclassSegFormer_Unified(backbone='MiT-B0', num_classes=2, subclass=64, num_prompts=5)
if os.path.exists(PRETRAINED_WEIGHT_PATH):
    checkpoint = torch.load(PRETRAINED_WEIGHT_PATH, map_location='cpu', weights_only=False)
    state_dict = checkpoint.get('state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint.state_dict()
    if 'decode_head.subclass_block.q_prime' in state_dict:
        del state_dict['decode_head.subclass_block.q_prime']
    model.load_state_dict(state_dict, strict=False)
model = model.to(DEVICE).eval()
print("✅ 環境與模型初始化完成。")

In [16]:
import os
import json
import torch
import numpy as np
from PIL import Image, ImageDraw
from pathlib import Path
from collections import Counter
from tqdm import tqdm
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision.utils import make_grid, save_image
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import euclidean_distances

from semseg.models.subclass_segformer_unified import SubclassSegFormer_Unified

# ================= 設定區域 =================
# 1. 訓練場域 (A-E) 原始路徑
TRAIN_ROOT_PATH = r"C:\Users\user\PycharmProjects\organized_ripple_4fold"
TRAIN_FIELDS = ['barrel', 'sea', 'LNG', 'seabass_hmh', 'noon_jsj']

# 2. 測試場域 (F) 列表路徑
TEST_TXT_PATH = r"/folds_experiment/fold_0/F_mask_hard_samples_val_list.txt"

# 3. 模型與輸出設定
PRETRAINED_WEIGHT_PATH = r"/output_mix5_SubclassSegFormer_Unified/SubclassSegFormer_Unified_MiT-B0_ripple.pth"
MEDOID_SAVE_DIR = "prompt_256D"
OUTPUT_DIR = "vlm_knowledge_construction_new_cluster"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MEDOID_SAVE_DIR, exist_ok=True)

RANGE_N_CLUSTERS = [6, 8, 10, 12, 16] # K 值搜尋範圍
SIMILARITY_THRESHOLD = 0.80 # OOD 判定門檻
IMAGE_SIZE = [800, 800]
PATCH_SIZE = 64
ANCHOR_SIZE = 256
GLOBAL_RESIZE = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 4. Expert Knowledge Injection 字典 (整合 A-E 與新的 F 場域設定)
FIELD_INTENSITY_RANGE = {
    "F": (6, 8)
}

FIELD_METADATA_MAPPING = {
    "IMG6235": {
        "field_id": "F",
        "intensity_baseline": 6,
        "environment": "offshore_cage",
        "fish_species": ["pompano"],
        "perspective": "medium_oblique"
    }
}
# ============================================

def calculate_intensity_stats(source_counts, total_count):
    weighted_sum = 0.0
    min_scores, max_scores = [], []
    for field_id, count in source_counts.items():
        weight = count / total_count
        baseline = next((m['intensity_baseline'] for m in FIELD_METADATA_MAPPING.values() if m['field_id'] == field_id), 0)
        weighted_sum += baseline * weight
        if weight > 0.1 and field_id in FIELD_INTENSITY_RANGE:
            min_scores.append(FIELD_INTENSITY_RANGE[field_id][0])
            max_scores.append(FIELD_INTENSITY_RANGE[field_id][1])
    return int(round(weighted_sum)), min(min_scores) if min_scores else 0, max(max_scores) if max_scores else 0

def load_custom_model(weight_path):
    print(">>> 載入統一版模型骨架 (SubclassSegFormer_Unified)...")
    model = SubclassSegFormer_Unified(backbone='MiT-B0', num_classes=2, subclass=64, num_prompts=5)
    if os.path.exists(weight_path):
        checkpoint = torch.load(weight_path, map_location='cpu', weights_only=False)
        state_dict = checkpoint.get('state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint.state_dict()
        q_prime_key = 'decode_head.subclass_block.q_prime'
        if q_prime_key in state_dict:
            if state_dict[q_prime_key].shape != model.decode_head.subclass_block.q_prime.shape:
                del state_dict[q_prime_key]
        model.load_state_dict(state_dict, strict=False)
        print(f"✅ 模型權重載入完成: {weight_path}")
    model = model.to(DEVICE).eval()
    return model

def extract_gap_feature(img_path, lbl_path, model):
    img_pil = Image.open(img_path).convert('RGB')
    image = TF.to_tensor(img_pil)
    img_resized = TF.resize(image, IMAGE_SIZE, interpolation=TF.InterpolationMode.BILINEAR)
    img_norm = TF.normalize(img_resized, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        features = model.backbone(img_norm)
        seg_map = model.decode_head.get_fusion_map(features)
    feature_map = seg_map.squeeze(0).cpu()

    label_np = np.array(Image.open(lbl_path).convert('L'))
    label_tensor = torch.tensor((label_np > 0).astype(np.uint8)).unsqueeze(0).unsqueeze(0).float()
    label_resized = F.interpolate(label_tensor, size=feature_map.shape[1:], mode='nearest').squeeze().long()
    ripple_mask = (label_resized == 1)

    if ripple_mask.sum() == 0:
        return None, None
    return feature_map[:, ripple_mask].mean(dim=1).numpy(), image

def get_crop_and_bbox(img_tensor, center_coord, crop_size):
    _, H, W = img_tensor.shape
    x_center, y_center = center_coord
    top = max(0, y_center - crop_size // 2)
    left = max(0, x_center - crop_size // 2)
    if top + crop_size > H: top = H - crop_size
    if left + crop_size > W: left = W - crop_size

    patch = TF.crop(img_tensor, top, left, crop_size, crop_size)
    if patch.shape[1] != crop_size or patch.shape[2] != crop_size:
        patch = TF.resize(patch, [crop_size, crop_size])
    return patch, (left, top, left + crop_size, top + crop_size)

def step1_build_knowledge_base(model):
    print("\n" + "="*50)
    print(" 步驟 1/3：提取 A-E 訓練集特徵 (建立最新知識庫)")
    print("="*50)

    all_features_list = []
    for field in TRAIN_FIELDS:
        img_dir = Path(TRAIN_ROOT_PATH) / field / 'images'
        lbl_dir = Path(TRAIN_ROOT_PATH) / field / 'labels_detectron2'

        if not img_dir.exists() or not lbl_dir.exists(): continue

        img_files = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
        print(f"處理 {field} ({len(img_files)} 張)...")

        for img_path in tqdm(img_files, leave=False):
            lbl_path = lbl_dir / f"{img_path.stem}.png"
            if not lbl_path.exists(): continue
            feat, _ = extract_gap_feature(img_path, lbl_path, model)
            if feat is not None:
                all_features_list.append(feat)

    all_features_np = np.vstack(all_features_list)
    return all_features_np

def step2_run_kmeans(features_np):
    print("\n" + "="*50)
    print(" 步驟 2/3：自動尋找最佳 K 值 (Silhouette) 與質心提取")
    print("="*50)

    features_norm = normalize(features_np, norm='l2', axis=1)
    best_k = RANGE_N_CLUSTERS[0]
    best_score = -1

    sample_size = min(10000, features_norm.shape[0])
    indices = np.random.choice(features_norm.shape[0], sample_size, replace=False)
    sample_data = features_norm[indices]

    for n_clusters in RANGE_N_CLUSTERS:
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            clusterer = KMeans(n_clusters=n_clusters, random_state=3407, n_init='auto')
            cluster_labels = clusterer.fit_predict(sample_data)
        silhouette_avg = silhouette_score(sample_data, cluster_labels)
        print(f"K = {n_clusters}, Silhouette Score : {silhouette_avg:.4f}")
        if silhouette_avg > best_score:
            best_score, best_k = silhouette_avg, n_clusters

    print(f"===> 最佳 K 值為: {best_k}")

    kmeans = KMeans(n_clusters=best_k, random_state=3407, n_init=10)
    labels = kmeans.fit_predict(features_norm)

    medoids = []
    for i in range(best_k):
        cluster_points = features_norm[np.where(labels == i)[0]]
        centroid = kmeans.cluster_centers_[i].reshape(1, -1)
        min_dist_idx = np.argmin(euclidean_distances(cluster_points, centroid))
        medoids.append(cluster_points[min_dist_idx])

    medoids_tensor = torch.tensor(np.array(medoids)).float()
    save_path = os.path.join(MEDOID_SAVE_DIR, f"visual_bases_K{best_k}_medoid_NEW.pth")
    torch.save(medoids_tensor, save_path)
    return medoids_tensor.numpy(), best_k

def step3_evaluate_and_expand_F(model, existing_medoids):
    print("\n" + "="*50)
    print(" 步驟 3/3：測試 F 場域並進行 OOD 擴充")
    print("="*50)

    features_list = []
    image_metadata = []

    with open(TEST_TXT_PATH, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    for line in lines:
        if not line.strip(): continue
        parts = line.split('|')
        img_name = parts[0].replace('IMG:', '').strip()
        label_path = parts[1].replace('PATH:', '').strip()

        # 修正 1：原圖與標註在同一資料夾，直接使用 label_path，不再替換 image
        rgb_path = label_path

        if not os.path.exists(rgb_path) or not os.path.exists(label_path): continue

        feat, img_tensor = extract_gap_feature(rgb_path, label_path, model)
        if feat is None: continue

        features_list.append(feat)

        label_np = np.array(Image.open(label_path).convert('L'))
        y_indices, x_indices = np.where(label_np > 0)
        y_center, x_center = (int(np.mean(y_indices)), int(np.mean(x_indices))) if len(y_indices) > 0 else (img_tensor.shape[1]//2, img_tensor.shape[2]//2)

        image_metadata.append({
            'img_name': img_name,
            # 修正 2：對應上方自訂的字典 Key，確保抓取到正確的屬性
            'field': 'IMG6235',
            'rgb_path': rgb_path,
            'center': (x_center, y_center),
            'img_tensor': img_tensor
        })

    if not features_list:
        print("❌ F 場域特徵提取失敗，請檢查資料路徑。")
        return

    features_norm = normalize(np.vstack(features_list), norm='l2', axis=1)

    distances = euclidean_distances(features_norm, existing_medoids)
    cosine_sims = 1 - (distances ** 2) / 2

    votes = []
    for sim in cosine_sims:
        if np.max(sim) < SIMILARITY_THRESHOLD:
            votes.append("OOD")
        else:
            votes.append("KNOWN")

    final_decision = Counter(votes).most_common(1)[0][0]

    if final_decision != "OOD":
        print(f"✅ F 場域被判定為已知特徵，無需新增。")
        return

    print(f"🚨 觸發 OOD 警報！啟動自動擴充程序：建構全新 Cluster [New_Field_F]...")

    f_centroid = np.mean(features_norm, axis=0).reshape(1, -1)
    dists_to_f_centroid = euclidean_distances(features_norm, f_centroid).flatten()
    medoid_idx = np.argmin(dists_to_f_centroid)
    medoid_info = image_metadata[medoid_idx]

    top_m_indices = np.argsort(dists_to_f_centroid)[:min(25, len(features_norm))]

    source_stats = [FIELD_METADATA_MAPPING.get(image_metadata[idx]['field'], {}).get('field_id', 'Unknown') for idx in top_m_indices]
    source_counts = Counter(source_stats)
    total_count = len(top_m_indices)
    dist_str = ", ".join([f"{key}({val/total_count:.1%})" for key, val in source_counts.most_common()])
    dominant_field_id = source_counts.most_common(1)[0][0]
    dominant_meta = next((m for m in FIELD_METADATA_MAPPING.values() if m['field_id'] == dominant_field_id), {})
    avg_score, range_min, range_max = calculate_intensity_stats(source_counts, total_count)

    anchor_patch, bbox = get_crop_and_bbox(medoid_info['img_tensor'], medoid_info['center'], ANCHOR_SIZE)
    medoid_pil = TF.to_pil_image(medoid_info['img_tensor'])
    draw = ImageDraw.Draw(medoid_pil)
    draw.rectangle(bbox, outline="red", width=8)
    context_img = TF.to_tensor(medoid_pil.resize((GLOBAL_RESIZE, GLOBAL_RESIZE)))

    variance_patches = []
    for idx in top_m_indices:
        patch, _ = get_crop_and_bbox(image_metadata[idx]['img_tensor'], image_metadata[idx]['center'], PATCH_SIZE)
        variance_patches.append(patch)

    variance_grid = make_grid(torch.stack(variance_patches), nrow=5, padding=2, normalize=False)
    variance_grid_resized = TF.resize(variance_grid, [GLOBAL_RESIZE, GLOBAL_RESIZE])

    final_dashboard = torch.cat([context_img, TF.resize(anchor_patch, [GLOBAL_RESIZE, GLOBAL_RESIZE]), variance_grid_resized], dim=2)
    dashboard_filename = "cluster_New_F_dashboard.jpg"
    save_image(final_dashboard, os.path.join(OUTPUT_DIR, dashboard_filename))

    new_cluster_draft = [{
        "cluster_id": "New_Field_F",
        "visual_dashboard_path": dashboard_filename,
        "expert_metadata": {
            "dominant_field": dominant_field_id,
            "source_distribution": dist_str,
            "environment_type": dominant_meta.get('environment', 'Unknown'),
            "fish_species": dominant_meta.get('fish_species', 'Unknown'),
            "perspective": dominant_meta.get('perspective', 'Unknown'),
            "splash_intensity": avg_score,
            "intensity_min": range_min,
            "intensity_max": range_max
        },
        "vlm_tasks": {
            "water_color": "TODO: Select from [dark_blue, greenish, muddy_brown, grey_concrete, black_monochrome]",
            "texture_type": "TODO: Select from [smooth_concentric, chaotic_ripples, foamy_white, glassy, striated_noise]",
            "splash_shape": "TODO: Select from [droplets, columnar, ripple, spray, boiling, chaotic_whitewater, faint_disturbance]",
            "surface_cover": "TODO: Select from [none, white_netting_overlay, floating_algae, foam_scum]",
            "interference": "TODO: Select from [none, paddlewheel_aerator, aerator_bubbles, bird_or_rat, plastic_pipes, green_net_fencing, distant_cages]",
            "lighting": "TODO: Select from [diffuse, high_glare, shadowed, infrared_night_vision, ir_stripes]",
            "container_edge": "TODO: Select from [none, open_water, plastic_edge, concrete_wall, netting]",
            "description": "TODO: Provide concise visual description for this newly discovered cluster."
        }
    }]

    json_path = os.path.join(OUTPUT_DIR, "cluster_New_F_knowledge_draft.json")
    with open(json_path, "w", encoding='utf-8') as f:
        json.dump(new_cluster_draft, f, indent=4, ensure_ascii=False)

    print(f"📝 JSON 填寫範本已產出: {json_path}")

In [17]:
if __name__ == "__main__":
    model = load_custom_model(PRETRAINED_WEIGHT_PATH)
    features_ae = step1_build_knowledge_base(model)
    new_medoids, _ = step2_run_kmeans(features_ae)

>>> 載入統一版模型骨架 (SubclassSegFormer_Unified)...
✅ 模型權重載入完成: C:\Users\user\PycharmProjects\subclass_segformer\output_mix5_SubclassSegFormer_Unified\SubclassSegFormer_Unified_MiT-B0_ripple.pth

 步驟 1/3：提取 A-E 訓練集特徵 (建立最新知識庫)
處理 barrel (500 張)...


處理 sea (500 張)...


處理 LNG (500 張)...


處理 seabass_hmh (500 張)...


處理 noon_jsj (500 張)...



 步驟 2/3：自動尋找最佳 K 值 (Silhouette) 與質心提取
K = 6, Silhouette Score : 0.2926
K = 8, Silhouette Score : 0.2575
K = 10, Silhouette Score : 0.2263
K = 12, Silhouette Score : 0.2281
K = 16, Silhouette Score : 0.2207
===> 最佳 K 值為: 6


C:\Users\user\anaconda3\envs\subclass_5090\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=10.
  warnings.warn(


In [18]:
    step3_evaluate_and_expand_F(model, new_medoids)


 步驟 3/3：測試 F 場域並進行 OOD 擴充
🚨 觸發 OOD 警報！啟動自動擴充程序：建構全新 Cluster [New_Field_F]...
📝 JSON 填寫範本已產出: vlm_knowledge_construction_new_cluster\cluster_New_F_knowledge_draft.json


In [1]:
import json
import os

# ================= 設定區域 =================
INPUT_DIR = "vlm_knowledge_construction_new_cluster"
INPUT_JSON = os.path.join(INPUT_DIR, "cluster_New_F_knowledge_draft.json")

def main():
    if not os.path.exists(INPUT_JSON):
        print(f"錯誤: 找不到 {INPUT_JSON}")
        return

    with open(INPUT_JSON, "r", encoding='utf-8') as f:
        data = json.load(f)

    print("========================================================")
    print(f" Gemini 3.0 Advanced 專用 Prompt 生成器 (256D 特徵對齊版)")
    print("========================================================\n")

    for entry in data:
        c_id = entry['cluster_id']
        img_path = entry['visual_dashboard_path']
        meta = entry['expert_metadata']

        avg_intensity = meta.get('splash_intensity', 'Unknown')
        min_int = meta.get('intensity_min', 'Unknown')
        max_int = meta.get('intensity_max', 'Unknown')

        intensity_context = f"Average: {avg_intensity}/10 (Range: {min_int}-{max_int})"

        # [新增] 讀取專家已知的視角資訊
        perspective_context = meta.get('perspective', 'Unknown')

        print(f"### Cluster {c_id} (請上傳圖片: {img_path}) ###")
        print("-" * 20 + " 複製下方文字 " + "-" * 20)

        # 整合了最新論文發現與 256D 物理意義的 Prompt
        prompt = f"""
I am analyzing aquaculture water splash patterns mapped in a 256D Continuous Feature Space. Act as a Computer Vision Expert.

I have provided a "Visual Dashboard" composed of:
1. **LEFT (Context):** A global view (Medoid of the cluster).
2. **CENTER (Anchor):** A close-up prototype of the splash.
3. **RIGHT (Variance):** 25 random sample patches from this specific cluster.

---
[HARD CONSTRAINTS] (Expert Metadata - These are GROUND TRUTHS, do not alter them)
* Environment: {meta.get('environment_type')}
* Source Field: {meta.get('dominant_field')} ({meta.get('source_distribution')})
* Fish Species: {meta.get('fish_species')}
* **Camera Perspective: {perspective_context}** * **Calculated Intensity Profile: {intensity_context}**
  (Note: This intensity is statistically derived from field pixel data. Do NOT guess the score, but describe features matching this intensity.)
---

[DOMAIN KNOWLEDGE RULES]
1. **Imaging Modality:**
   - Check for Infrared (black/white/reddish tint, stripes) vs Visible Light.
2. **Color & Environment:**
   - **Offshore/Sea:** Dark blue, Black.
   - **Onshore Ponds:** Greenish (Algae), Muddy Brown, Grey (Concrete).
3. **Visual Texture Mapping:**
   - **High Intensity (7-10):** Look for "Boiling", "Chaotic Whitewater".
   - **Medium Intensity (4-6):** Look for "Droplets", "Spray", "Ripple".
   - **Low Intensity (1-3):** Look for "Glassy Surface", "Faint Disturbance".
4. **Interference Awareness:**
   - Differentiate between "Paddlewheel Aerator" (large mechanical splashes) and "Aerator Bubbles" (fine underwater bubbles).

---
[TASK: VISUAL TEXTURE ANALYSIS]
Observe the Anchor and Variance images. Generate a structured JSON describing this cluster's pure visual semantics.
**You MUST strictly choose from the provided lists.**

```json
{{
    "water_color": "Select one: [dark_blue, greenish, muddy_brown, grey_concrete, black_monochrome]",
    "texture_type": "Select one: [smooth_concentric, chaotic_ripples, foamy_white, glassy, striated_noise]",
    "splash_shape": "Select one: [droplets, columnar, ripple, spray, boiling, chaotic_whitewater, faint_disturbance]",
    "surface_cover": "Select one: [none, white_netting_overlay, floating_algae, foam_scum]",
    "interference": "Select one: [none, paddlewheel_aerator, aerator_bubbles, bird_or_rat, plastic_pipes, green_net_fencing, distant_cages]",
    "lighting": "Select one: [diffuse, high_glare, shadowed, infrared_night_vision, ir_stripes]",
    "container_edge": "Select one: [none, open_water, plastic_edge, concrete_wall, netting]",
    "description": "A concise sentence describing the visual appearance, explicitly mentioning the water texture, lighting, and splash dynamics."
}}
```"""
        print(prompt.strip())
        print("-" * 55)
        print("\n\n")

if __name__ == "__main__":
    main()

 Gemini 3.0 Advanced 專用 Prompt 生成器 (256D 特徵對齊版)

### Cluster 6 (請上傳圖片: cluster_New_F_dashboard.jpg) ###
-------------------- 複製下方文字 --------------------
I am analyzing aquaculture water splash patterns mapped in a 256D Continuous Feature Space. Act as a Computer Vision Expert.

I have provided a "Visual Dashboard" composed of:
1. **LEFT (Context):** A global view (Medoid of the cluster).
2. **CENTER (Anchor):** A close-up prototype of the splash.
3. **RIGHT (Variance):** 25 random sample patches from this specific cluster.

---
[HARD CONSTRAINTS] (Expert Metadata - These are GROUND TRUTHS, do not alter them)
* Environment: offshore_cage
* Source Field: F (F(100.0%))
* Fish Species: ['pompano']
* **Camera Perspective: medium_oblique** * **Calculated Intensity Profile: Average: 6/10 (Range: 6-8)**
  (Note: This intensity is statistically derived from field pixel data. Do NOT guess the score, but describe features matching this intensity.)
---

[DOMAIN KNOWLEDGE RULES]
1. **Imaging Modali